In [3]:
!pip install pandas numpy
import pandas as pd
import numpy  as np

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: C:\Users\bende\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


**step one get the cst file and watch it**

In [46]:
df = pd.read_csv('events_raw.csv')
df_starting_len =len(df)
df.head()

,event_id,timestamp,event_type,user_id,status,latency_ms
0,928,2026-01-15 21:39:53,checkout,user_123,error,316
1,631,2026-01-15 19:35:45,purchase,user_253,ok,1652
2,683,2026-01-15 01:07:47,purchase,user_154,error,1543
3,515,2026-01-15 10:26:46,view,user_257,error,733
4,366,2026-01-15 20:11:32,checkout,user_925,error,588


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1006 entries, 0 to 1005
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   event_id    1006 non-null   int64
 1   timestamp   1006 non-null   str  
 2   event_type  961 non-null    str  
 3   user_id     985 non-null    str  
 4   status      975 non-null    str  
 5   latency_ms  945 non-null    str  
dtypes: int64(1), str(5)
memory usage: 47.3 KB


In [6]:
df.isna().sum()

event_id       0
timestamp      0
event_type    45
user_id       21
status        31
latency_ms    61
dtype: int64

**key problems**

- event_type
- user_id
- status
- latency_ms

all has some na values
---------------------------------------
another problems
- timestams
- latency_ms
- maybe user id

not in the right type





In [7]:
df['event_type'].value_counts(dropna=False)

event_type
view            241
checkout        235
login           234
purchase        222
NaN              45
invalid_type     29
Name: count, dtype: int64

In [8]:
df['status'].value_counts(dropna=False)

status
error      337
timeout    322
ok         297
NaN         31
failed      19
Name: count, dtype: int64

In [9]:
df['latency_ms'].value_counts(dropna=False)

latency_ms
NaN        61
timeout    10
643         4
865         4
1918        4
           ..
877         1
1853        1
1675        1
1659        1
1342        1
Name: count, Length: 730, dtype: int64

--------------------
**part 2**
-------------

**4**

In [10]:
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df['latency_ms'] = pd.to_numeric(df['latency_ms'], errors='coerce')
df['event_id'] = df['event_id'].astype('int64')

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1006 entries, 0 to 1005
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   event_id    1006 non-null   int64         
 1   timestamp   1006 non-null   datetime64[us]
 2   event_type  961 non-null    str           
 3   user_id     985 non-null    str           
 4   status      975 non-null    str           
 5   latency_ms  935 non-null    float64       
dtypes: datetime64[us](1), float64(1), int64(1), str(3)
memory usage: 47.3 KB


**5**

In [12]:
df.isna().sum()

event_id       0
timestamp      0
event_type    45
user_id       21
status        31
latency_ms    71
dtype: int64

For critical columns (event_id, timestamp, event_type):
Remove rows with missing values


**pre droping**

In [13]:
pre_drop = len(df)

In [14]:
df = df.dropna(subset=['event_id', 'timestamp', 'event_type'],axis=0)

In [15]:
f"droped {pre_drop - len(df)} events    "

'droped 45 events    '

For optional columns (status, latency_ms):
Fill missing values with reasonable defaults


In [16]:
df['status'] = df['status'].fillna('unknown')
df['latency_ms']=df['latency_ms'].fillna(df['latency_ms'].mean() )


Step 6: Remove Duplicates

In [17]:
df[df.duplicated()]

,event_id,timestamp,event_type,user_id,status,latency_ms
260,6,2026-01-15 08:07:36,checkout,user_105,timeout,1661.0
376,7,2026-01-15 05:04:49,view,user_826,error,1377.0
426,10,2026-01-15 03:09:55,login,user_892,error,1475.0
563,11,2026-01-15 21:29:57,view,user_730,ok,708.0
801,9,2026-01-15 19:51:22,view,user_430,timeout,1906.0
904,8,2026-01-15 03:43:54,view,user_140,ok,1550.0


so it looks like there are some duplicated therefore i will check the amount than drop them

In [18]:
duplicate_amount = len(df[df.duplicated()])
duplicate_amount

6

In [19]:
df_supposed_len = len(df) - duplicate_amount

In [20]:
df = df.drop_duplicates()

len(df) == df_supposed_len


True

**step 7**

In [21]:
uniq_events = df['event_type'].unique()

uniq_events

<StringArray>
['checkout', 'purchase', 'view', 'login', 'invalid_type']
Length: 5, dtype: str

normalizing "invalid_type" event 

In [22]:
df.loc[df['event_type'] == 'invalid_type', 'event_type'] = 'unknown'

df['event_type'].unique()

<StringArray>
['checkout', 'purchase', 'view', 'login', 'unknown']
Length: 5, dtype: str

**step 8**

In [23]:
uniq_status = df['status'].unique()

uniq_status


<StringArray>
['error', 'ok', 'timeout', 'unknown', 'failed']
Length: 5, dtype: str

In [24]:
valid_statuses = set(["ok", "error", "timeout", "unknown"] )
valid_statuses

{'error', 'ok', 'timeout', 'unknown'}

In [25]:
df[~(df['status'].isin(valid_statuses))].value_counts()

event_id  timestamp            event_type  user_id   status  latency_ms
985       2026-01-15 11:27:04  unknown     user_903  failed  502.0         1
78        2026-01-15 22:10:40  purchase    user_749  failed  228.0         1
589       2026-01-15 10:05:47  view        user_540  failed  1040.0        1
582       2026-01-15 16:14:06  checkout    user_663  failed  469.0         1
936       2026-01-15 09:54:43  view        user_737  failed  1752.0        1
996       2026-01-15 01:58:24  view        user_211  failed  631.0         1
750       2026-01-15 09:46:19  view        user_133  failed  1984.0        1
590       2026-01-15 18:58:47  checkout    user_431  failed  1169.0        1
766       2026-01-15 17:45:04  purchase    user_821  failed  960.0         1
125       2050-01-15 10:37:34  checkout    user_301  failed  233.0         1
846       2026-01-15 17:05:24  checkout    user_211  failed  1151.0        1
724       2026-01-15 15:35:48  unknown     user_888  failed  953.0         1
984 

**step 9**

In [26]:
df.describe()

,event_id,timestamp,latency_ms
count,955.000000,955,955.000000
mean,499.531937,2026-07-18 01:57:26.361256,1012.571687
min,1.000000,2026-01-15 00:00:53,50.000000
25%,252.500000,2026-01-15 06:23:44,580.500000
50%,500.000000,2026-01-15 12:01:19,1015.278835
75%,746.500000,2026-01-15 18:18:07,1473.500000
max,1000.000000,2050-01-15 23:34:56,2000.000000
std,288.034843,NaN,541.218537


In [27]:
df.isnull().sum()

event_id       0
timestamp      0
event_type     0
user_id       19
status         0
latency_ms     0
dtype: int64

In [28]:
# Check timestamp range
print("\n\nTimestamp Range:")
print(f"Earliest: {df['timestamp'].min()}")
print(f"Latest: {df['timestamp'].max()}")



Timestamp Range:
Earliest: 2026-01-15 00:00:53
Latest: 2050-01-15 23:34:56


**10**

In [29]:
difference_row_amount = df_starting_len - len(df)
difference_row_amount

51

In [30]:
print(f"Data retained:         {len(df)/df_starting_len*100:.1f}%")
print(f"Data removed:          {(1 - len(df)/df_starting_len)*100:.1f}%")


Data retained:         94.9%
Data removed:          5.1%


part 4 
----

**step 11**

In [31]:

error_amount = (df['status'] == 'error').sum()

error_amount


np.int64(320)

In [32]:
error_rate = (error_amount/len(df)) *100

error_rate

np.float64(33.50785340314136)

In [33]:
success_amount = len(df) - error_amount

success_amount

np.int64(635)

In [34]:
# event type

df['status'].value_counts()

status
error      320
timeout    306
ok         281
unknown     29
failed      19
Name: count, dtype: int64

In [35]:
df.groupby('status')['latency_ms'].mean()

status
error      1014.173677
failed     1143.842105
ok          995.482479
timeout    1010.391150
unknown    1097.486736
Name: latency_ms, dtype: float64

**12**

In [36]:
group_by_e_t = df.groupby('event_type')

error_counts = group_by_e_t['status'].value_counts()[:,'error']

error_counts

event_type
checkout    77
login       77
purchase    79
unknown      8
view        79
Name: count, dtype: int64

In [37]:
top_events = error_counts[error_counts == error_counts.max()].index.tolist()
top_events

['purchase', 'view']

find the slowest 

In [38]:
group_by_e_t['latency_ms'].mean().idxmax()

'login'

most common event type

In [39]:
group_by_e_t.count().idxmax()

event_id      view
timestamp     view
user_id       view
status        view
latency_ms    view
dtype: str

Part 5: Save and Present
----
**13**

In [40]:
# df.to_csv('events_cleaned.csv', index=False)

In [45]:
print("\n\nFinal Data Quality Report:")
print("=" * 40)
print(f"Total events: {len(df):,}")
print(f"Complete records: {(df.isnull().sum() == 0).sum()} / {len(df.columns)} columns")
print(f"Error rate: {error_rate:.2f}%")
# print(f"Slowest event: {avg_latency_by_type.index[0]}")
print(f"Data quality: {'✓ GOOD' if df_starting_len - len(df) < df_starting_len * 0.2 else '⚠ CHECK'}")





Final Data Quality Report:
Total events: 955
Complete records: 5 / 6 columns
Error rate: 33.51%
Data quality: ✓ GOOD
